In [1]:
import os
import torch
import json
import shutil
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# AI Libraries
from ultralytics import YOLO
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.deploy import TorchInferencer

# --- 1. PATH CONFIGURATION (Based on your Screenshot) ---
PROJECT_ROOT = Path("E:/Project-Work")

# Inputs
RAW_DATA_DIR = PROJECT_ROOT / "Data/TOP 1"
ROI_CONFIG_PATH = PROJECT_ROOT / "Preprocessing/roi_config.json"
LABELS_CSV = PROJECT_ROOT / "Preprocessing/features_labeled.csv"
REFERENCE_IMG_NAME = "20251106131917_TOP.png" # The master image for alignment

# Outputs (We will create these)
ALIGNED_DIR = PROJECT_ROOT / "Data/aligned_top"
CROPS_DIR = PROJECT_ROOT / "Data/connectors"
FINAL_DATASET_DIR = PROJECT_ROOT / "Final_Dataset"

# Make sure outputs exist
for d in [ALIGNED_DIR, CROPS_DIR, FINAL_DATASET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Project Root set to: {PROJECT_ROOT}")

✅ Project Root set to: E:\Project-Work


In [2]:
def load_image(path):
    path = str(path)
    img = cv2.imread(path)
    if img is None: raise FileNotFoundError(f"Cannot read: {path}")
    return img

def align_image(img, ref_img, ref_gray, crop_coords):
    """Aligns board to reference and crops the background"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # ORB Features
    orb = cv2.ORB_create(nfeatures=5000)
    kp1, des1 = orb.detectAndCompute(ref_gray, None)
    kp2, des2 = orb.detectAndCompute(gray, None)
    
    # Match Features
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = matcher.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    
    # Keep top 15% matches
    good_matches = matches[:int(len(matches) * 0.15)]
    if len(good_matches) < 10: return None 
    
    # Extract points
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    # Find Homography & Warp
    H, _ = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    h, w = ref_img.shape[:2]
    warped = cv2.warpPerspective(img, H, (w, h))
    
    # Apply the "Background Removal" Crop (x1, y1, x2, y2)
    x1, y1, x2, y2 = crop_coords
    return warped[y1:y2, x1:x2]

def get_connector_crop(aligned_img, roi_config, margin=8):
    """Cuts out specific connector based on JSON percentages"""
    h, w = aligned_img.shape[:2]
    
    x1 = int(roi_config['x_min_rel'] * w) - margin
    y1 = int(roi_config['y_min_rel'] * h) - margin
    x2 = int(roi_config['x_max_rel'] * w) + margin
    y2 = int(roi_config['y_max_rel'] * h) + margin
    
    # Safety check boundaries
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    
    return aligned_img[y1:y2, x1:x2]

In [ ]:
# 1. Load Reference Image
ref_path = RAW_DATA_DIR / REFERENCE_IMG_NAME
ref_img = load_image(ref_path)
ref_gray = cv2.cvtColor(ref_img, cv2.COLOR_BGR2GRAY)

# 2. Define the Background Crop (The fix we discussed)
# Coordinates: x1, y1, x2, y2
BG_CROP_BOX = (302, 288, 1883, 942)

# 3. Load ROI JSON
with open(ROI_CONFIG_PATH, 'r') as f:
    roi_configs = json.load(f)

# 4. Process Loop
all_images = list(RAW_DATA_DIR.glob("*.png"))
print(f"🚀 Processing {len(all_images)} images...")

for img_path in tqdm(all_images):
    try:
        # Step A: Align & Background Crop
        img = load_image(img_path)
        aligned = align_image(img, ref_img, ref_gray, BG_CROP_BOX)
        
        if aligned is None:
            print(f"⚠️ Alignment failed: {img_path.name}")
            continue
            
        # Save aligned image (optional, good for debug)
        cv2.imwrite(str(ALIGNED_DIR / img_path.name), aligned)
        
        # Step B: Cut out Connectors
        for roi in roi_configs:
            crop = get_connector_crop(aligned, roi)
            
            # Ensure folder exists: Data/connectors/conn1/
            save_folder = CROPS_DIR / roi['name']
            save_folder.mkdir(exist_ok=True)
            
            # Save crop
            cv2.imwrite(str(save_folder / img_path.name), crop)
            
    except Exception as e:
        print(f"❌ Error processing {img_path.name}: {e}")

print("✅ Preprocessing Done! Crops are in Data/connectors/")

In [48]:
import shutil

# Outputs
INSPECTOR_DATA = FINAL_DATASET_DIR / "inspector_by_connector"
CROPS_DIR = PROJECT_ROOT / "Data/connectors"  # Ensure this matches your paths

# 1. Clean and Recreate Structure
if INSPECTOR_DATA.exists():
    shutil.rmtree(INSPECTOR_DATA)

# 2. Read Labels
df = pd.read_csv(LABELS_CSV)
print(f"📄 Sorting {len(df)} images into Connector folders...")

# 3. Sort Files
for _, row in tqdm(df.iterrows(), total=len(df)):
    connector = row['connector_name']
    label = str(row['label']).upper().strip()
    filename = row['filename']
    
    src_path = CROPS_DIR / connector / filename
    if not src_path.exists(): continue
    
    # Destination Paths
    conn_base = INSPECTOR_DATA / connector
    good_dir = conn_base / "good"
    bad_dir = conn_base / "bad"
    
    # Ensure folders exist immediately (Fixes conn8 crash)
    good_dir.mkdir(parents=True, exist_ok=True)
    bad_dir.mkdir(parents=True, exist_ok=True)
    
    unique_name = f"{connector}_{filename}"
    
    if label == 'OK':
        shutil.copy(src_path, good_dir / unique_name)
    elif label == 'KO':
        shutil.copy(src_path, bad_dir / unique_name)

print("✅ Dataset repaired. All 'bad' folders exist now.")

📄 Sorting 1557 images into Connector folders...


  0%|          | 0/1557 [00:00<?, ?it/s]

✅ Dataset repaired. All 'bad' folders exist now.


In [ ]:
model = YOLO('yolov8n-cls.pt')

# Train
results = model.train(
    data=str(FINAL_DATASET_DIR / "gatekeeper_train"),
    epochs=50,
    imgsz=64,
    early_stopping=True,
    project="runs/classify",
    name="gatekeeper_notebook_run",
    exist_ok=True # Overwrite previous run
)
print("✅ Gatekeeper Trained")

New https://pypi.org/project/ultralytics/8.3.232 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.231  Python-3.10.19 torch-2.9.1+cpu CPU (AMD Ryzen 7 5800H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Project-Work\Final_Dataset\gatekeeper_train, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=gate

In [86]:
import shutil
import random

def augment_insufficient_data(conn_folder, min_bad=6):
    """
    If a connector has too few bad images, duplicate them to prevent
    ZeroDivisionError during splitting and ensure Test set gets at least 1.
    """
    bad_dir = conn_folder / "bad"
    if not bad_dir.exists():
        return 0

    bad_images = list(bad_dir.glob("*.png"))
    count = len(bad_images)
    
    # If we have 0 bad images, we can't do anything.
    if count == 0:
        return 0
        
    # If we have some, but less than limit, we inflate.
    if count < min_bad:
        print(f"   ⚠️ Inflating 'bad' data for stability (Original: {count})")
        needed = min_bad - count
        for i in range(needed):
            # Pick a random image to duplicate
            src = random.choice(bad_images)
            # Create a unique name: original_copy_1.png
            dst = bad_dir / f"{src.stem}_copy_{i}{src.suffix}"
            shutil.copy(src, dst)
        
        # Return new count
        return len(list(bad_dir.glob("*.png")))
    
    return count

def train_multi_inspector():
    # 1. Setup paths
    inspector_root = PROJECT_ROOT / "Final_Dataset/inspector_by_connector"
    connectors = [d for d in inspector_root.iterdir() if d.is_dir()]
    
    print(f"✅ Found {len(connectors)} connector types.")
    
    # Global settings
    SEED = 42
    performance_report = []
    
    for conn_folder in connectors:
        conn_name = conn_folder.name
        print(f"\n--- Processing: {conn_name} ---")
        
        # 2. Inflate Data (Prevents ZeroDivisionError)
        # We ensure we have enough 'bad' images for the split logic
        num_bad_final = augment_insufficient_data(conn_folder, min_bad=6)
        
        good_files = list((conn_folder / "good").glob("*.png"))
        num_good = len(good_files)
        
        print(f"   📊 Data: {num_good} good, {num_bad_final} bad images.")
        
        if num_good < 5:
            print(f"   ❌ Skipping {conn_name}: Not enough GOOD images.")
            continue

        # 3. Dynamic Batch Size
        # Lower batch size (8) to accommodate the larger WideResNet model
        safe_batch_size = 4 if num_good < 50 else 8 
        abnormal_arg = "bad" if num_bad_final > 0 else None
        
        # 4. Setup DataModule
        # REMOVED 'image_size' argument to fix TypeError
        datamodule = Folder(
            name=conn_name,
            root=str(conn_folder),
            normal_dir="good",
            abnormal_dir=abnormal_arg,
            num_workers=0, 
            train_batch_size=safe_batch_size,
            eval_batch_size=safe_batch_size,
            # test_split_ratio=0.2,
            val_split_ratio=0.2,
            seed=SEED
        )
        
        # 5. Setup Model (The Performance Upgrade)
        # We use WideResNet50 instead of ResNet18 for better accuracy
        # We use a higher sampling ratio (0.25) to keep more memory
        try:
            model = Patchcore(
                backbone="wide_resnet50_2", 
                pre_trained=True,
                coreset_sampling_ratio=0.1 
            )
        except Exception:
            print("   ⚠️ WideResNet failed (likely OOM), falling back to ResNet18")
            model = Patchcore(
                backbone="resnet18", 
                pre_trained=True,
                coreset_sampling_ratio=0.1
            )
        
        # 6. Setup Engine
        # We use 3 epochs to allow the threshold to stabilize
        engine = Engine(
            max_epochs=3, 
            default_root_dir=f"results/{conn_name}",
            accelerator="gpu",
            devices=1,
            num_sanity_val_steps=0,
            log_every_n_steps=1,
        )
        
        # 7. Train & Test
        try:
            engine.fit(datamodule=datamodule, model=model)
            
            if num_bad_final > 0:
                print(f"   🧪 Testing {conn_name}...")
                test_results = engine.test(datamodule=datamodule, model=model)
                try:
                    f1 = test_results[0].get('image_F1Score', 0.0)
                    auroc = test_results[0].get('image_AUROC', 0.0)
                except:
                    f1, auroc = 0.0, 0.0
            else:
                f1, auroc = -1, -1
                
            performance_report.append({'name': conn_name, 'F1': f1, 'AUROC': auroc})
            
            # 8. Export Model
            output_path = PROJECT_ROOT / f"Models/inspectors/{conn_name}_model_torch"
            output_path.parent.mkdir(parents=True, exist_ok=True)
            model.to_torch(export_root=output_path)
            print(f"   💾 Saved: {conn_name}")
            
        except Exception as e:
            print(f"   ❌ Failed processing {conn_name}: {e}")
            # import traceback
            # traceback.print_exc()

    # Final Report
    print("\n📋 SUMMARY 📋")
    print(f"{'Connector':<15} | {'F1':<10} | {'AUROC':<10}")
    for res in performance_report:
        f1_s = f"{res['F1']:.2f}" if res['F1'] != -1 else "N/A"
        auroc_s = f"{res['AUROC']:.2f}" if res['AUROC'] != -1 else "N/A"
        print(f"{res['name']:<15} | {f1_s:<10} | {auroc_s:<10}")

# Run it
train_multi_inspector()

✅ Found 9 connector types.

--- Processing: conn1 ---
   📊 Data: 171 good, 6 bad images.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
e:\miniconda3\envs\pw_clean\lib\site-packages\lightning\pytorch\core\optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer

  | Name           | Type           | Params | Mode 
----------------------------------------------------------
0 | pre_processor  | PreProcessor   | 0      | train
1 | post_processor | PostProcessor  | 0      | train
2 | evaluator      | Evaluator      | 0      | train
3 | model          | PatchcoreModel | 24.9 M | train
----------------------------------------------------------
24.9 M    Trainable params
0         Non-trainable params
24.9 M    Total params
99.450    Total estimated model params size (MB)
19        Modules in train mode
174       Modules in eval mode
e:\miniconda3\envs\pw_clean\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_datal

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]










































































































































































































































































































































































































































































































































































































































































































Selecting Coreset Indices.: 100%|██████████| 14028/14028 [01:15<00:00, 185.00it/s]
`Trainer.fit` stopped: `max_epochs=1` reached.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_R

...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]













































































































































































































































































































































































































































































































































































































































































































Selecting Coreset Indices.:  96%|█████████▌| 13499/14028 [01:19<00:03, 170.15it/s]

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

e:\miniconda3\envs\pw_clean\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [70]:
# --- Load Gatekeeper (Single Model) ---
gatekeeper_path = PROJECT_ROOT / "runs/classify/gatekeeper_notebook_run/weights/best.pt"
if not gatekeeper_path.exists():
    gatekeeper_path = "yolov8n-cls.pt" # Fallback
gk_model = YOLO(gatekeeper_path)

# --- Load Inspectors (Dictionary of Models) ---
inspector_models = {}
inspector_dir = PROJECT_ROOT / "Models/inspectors"

print("⏳ Loading Inspector Models...")
for model_folder in inspector_dir.iterdir():
    if model_folder.is_dir():
        # The naming convention from training was {conn_name}_model_torch
        # So we split the folder name to get the connector name back
        # Folder name example: "conn1_model_torch" -> name: "conn1"
        conn_name = model_folder.name.replace("_model_torch", "")
        
        model_file = model_folder / "weights/torch/model.pt"
        
        if model_file.exists():
            print(f"   Loading {conn_name}...")
            # We load it into memory. Since they are ResNet18 (small), 
            # holding 5-10 in VRAM should be fine. If OOM, we load on demand.
            inspector_models[conn_name] = TorchInferencer(
                path=model_file, 
                device="cuda"
            )

print(f"✅ Loaded {len(inspector_models)} Inspector models.")

⏳ Loading Inspector Models...
   Loading conn1...
   Loading conn2...


   Loading conn3...
   Loading conn4...
   Loading conn5...


   Loading conn6...
   Loading conn7...


   Loading conn8...
   Loading conn9...
✅ Loaded 9 Inspector models.


In [71]:
def visualize_prediction(image_path, gatekeeper, inspector):
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize
    
    # 1. Load Image
    img_path = str(image_path)
    image = cv2.imread(img_path)
    if image is None: 
        print(f"Could not read {img_path}")
        return
    # Convert BGR (OpenCV) to RGB (Matplotlib)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 

    # 2. Run Gatekeeper
    gk_result = gatekeeper(img_path, verbose=False)[0]
    gk_status = gk_result.names[gk_result.probs.top1].upper()
    gk_conf = gk_result.probs.top1conf.item()

    # 3. Run Inspector
    # TorchInferencer returns a generic Result object
    inspector_result = inspector.predict(image)
    
    # Extract maps and scores safely
    anomaly_map = inspector_result.anomaly_map
    pred_score = inspector_result.pred_score

    # --- FIX: ROBUST TENSOR TO NUMPY CONVERSION ---
    if isinstance(anomaly_map, torch.Tensor):
        anomaly_map = anomaly_map.detach().cpu().numpy()
    
    if isinstance(pred_score, torch.Tensor):
        score_val = pred_score.detach().cpu().item()
    else:
        score_val = float(pred_score)

    # 4. Create Visuals
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot A: Original Image
    axes[0].imshow(image)
    axes[0].set_title(f"Original\n{gk_status} ({gk_conf:.1%})")
    axes[0].axis("off")

    # Plot B: The Heatmap
    if anomaly_map is not None:
        # Squeeze dimensions: (1, 1, H, W) -> (H, W)
        anomaly_map = anomaly_map.squeeze()
        
        axes[1].imshow(anomaly_map, cmap="hot")
        axes[1].set_title(f"AI Attention Map\nScore: {score_val:.4f}")
    else:
        axes[1].text(0.5, 0.5, "No Map", ha='center')
    axes[1].axis("off")

    # Plot C: Overlay
    if anomaly_map is not None:
        h, w = image.shape[:2]
        # Resize heatmap to match image
        heatmap_resized = cv2.resize(anomaly_map, (w, h))
        
        # Normalize to 0-1
        min_val = heatmap_resized.min()
        max_val = heatmap_resized.max()
        
        # Avoid division by zero in normalization
        if max_val - min_val > 0:
            heatmap_norm = (heatmap_resized - min_val) / (max_val - min_val)
        else:
            heatmap_norm = heatmap_resized

        axes[2].imshow(image)
        axes[2].imshow(heatmap_norm, cmap="jet", alpha=0.5)

        # Verdict logic (Threshold is usually around 0.5 for Patchcore)
        verdict = "DISCONNECTED" if score_val > 0.5 else "CONNECTED"
        color = "red" if verdict == "DISCONNECTED" else "green"
        
        axes[2].set_title(f"Verdict: {verdict}", color=color, fontweight="bold")
    
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()